# Snake

In [ ]:
import environments_fully_observable 
import environments_partially_observable
from IPython.display import clear_output
import numpy as np
from  tqdm import trange
import matplotlib.pyplot as plt
import random
import tensorflow as tf
import os

from agent import Agent
import algorithms.actor_critic as ac
import algorithms.dqn as dqn
from replay_buffer import ReplayBuffer

tf.random.set_seed(0)
random.seed(0)
np.random.seed(0)

## Environment definition

In [ ]:
%matplotlib inline
# function to standardize getting an env for the whole notebook

N = 2000

def get_env(n=N):
    # n is the number of boards that you want to simulate parallely
    # size is the size of each board, also considering the borders
    # mask for the partially observable, is the size of the local neighborhood
    size = 7
    e = environments_fully_observable.OriginalSnakeEnvironment(n, size)
    # or environments_partially_observable.OriginalSnakeEnvironment(n, size, 2)
    return e
env_ = get_env()

GAMMA = .9
ITERATIONS = 100000

SAVE_FREQUENCY = 500
RESULTS_PATH = "results"


In [ ]:
fig,axs=plt.subplots(1,min(len(env_.boards), 5), figsize=(10,3))
for ax, board in zip(axs, env_.boards):
    ax.get_yaxis().set_visible(False)
    ax.get_xaxis().set_visible(False)
    ax.imshow(board, origin="lower")

## Model

In [ ]:
optimizer = tf.keras.optimizers.Adam(1e-4)

In [ ]:
# define the models that you need ()
#logic = ac.create_logic(state_shape=env_.to_state().shape[1:], action_dim=4, n_boards=len(env_.boards), optimizer=optimizer, gamma=GAMMA)

#logic.load_models(folder_path=os.path.join(RESULTS_PATH, "actor_critic_separated_loss_entropy", "best_model"), state_shape=env_.to_state().shape[1:], load_critic=False)

#wrapper for the agent that interacts with the environment, it will call the logic to get the action and to train the model
#agent = Agent(algorithm_logic=logic, algorithm_name="actor_critic_separated_loss_entropy", results_path=RESULTS_PATH, save_frequency=SAVE_FREQUENCY)

In [ ]:
logic = dqn.create_logic(state_shape=env_.to_state().shape[1:], action_dim=4, n_boards=len(env_.boards), optimizer=optimizer, gamma=GAMMA)
agent = Agent(algorithm_logic=logic, algorithm_name="dqn_99", results_path=RESULTS_PATH, save_frequency=SAVE_FREQUENCY)

## Training

In [ ]:
ITERS_UNTIL_STOP = 2000      # How many iterations to wait for improvement
MIN_DELTA = 0.0005   # Minimum improvement to reset patience
non_improving_iters = 0
best_moving_avg = -float('inf')
total_cumulative_reward = 0

buffer = ReplayBuffer(capacity=20000, state_shape=env_.to_state().shape[1:])
MIN_BUFFER_SIZE = 3000
BATCH_SIZE = 512

STEPS_PER_COLLECTION = 5
TRAIN_EPOCHS = 4

progress_bar = trange(ITERATIONS)

for iteration in progress_bar:

    for _ in range(STEPS_PER_COLLECTION):
        # get current state of the boards
        state = env_.to_state()
        #sample action
        actions, _ = agent.get_action(state, training=True)

        rewards = env_.move(actions)
        new_state = tf.cast(env_.to_state(), dtype=tf.float32)

        done = tf.cast(tf.math.abs(rewards - (-0.1)) < 1e-5, tf.float32)

        buffer.push(state, actions.numpy(), rewards.numpy(), new_state.numpy(), done.numpy())

        dead_boards_indices = np.where(done.numpy().flatten() == 1.0)[0]
        for b_idx in dead_boards_indices:
            # get_board() adds Walls and Head, but NO Fruit
            new_board = env_.get_board()
            
            # MANUALLY ADD FRUIT so move() doesn't crash next step
            empty = np.argwhere(new_board == env_.EMPTY)
            if len(empty) > 0:
                f = empty[np.random.choice(len(empty))]
                new_board[f[0], f[1]] = env_.FRUIT
                
            env_.boards[b_idx] = new_board
            env_.bodies[b_idx] = [] # Clear body list


    total_loss = 0
    for _ in range(TRAIN_EPOCHS):
        # Sample a batch from history
        s_b, a_b, r_b, ns_b, d_b = buffer.sample(BATCH_SIZE)
        
        # Train the agent on the sampled batch
        loss = agent.train_step(s_b, a_b, r_b, ns_b, d_b)
        total_loss += loss


    #clear the output of the cell to update the plot
    clear_output(wait=True)

        # Update progress bar
    if iteration % 20 == 0:
        recent_avg_reward = np.mean(agent.reward_history[-100:]) if agent.reward_history else 0
        progress_bar.set_description(f"Loss: {total_loss/TRAIN_EPOCHS:.4f} | Avg R: {recent_avg_reward:.3f}")

    if (iteration + 1) % SAVE_FREQUENCY == 0:
        agent.save_results_plots()